# Librerías

In [ ]:
import pandas as pd
from pathlib import Path

# Documents

Extraer los datos de los últimos 5 años (2020 - 2024):
Hojas de formación en las empresas, contexto y errores muestreo
Extraemos todos los datos respecto a formación en las empresas pero después analizaremos solo (en el caso de algunas tablas de formación cogeremos ya las específicas de servicios [e.g. 2024 EAL-23 y 23c]):
CCAA: Total y Cataluña y otras CCAA donde tenemos sede (Madrid, Valencia, Sevilla y Bilbao)
Tamaño empresa: más de 499
Sector Transporte y almacenamiento y cuando agregado servicios

TABLAS CONTEXTO
2020, 2021, 2022, 2023, 2024:
EAL-C1, EAL-C2, EAL-C3

TABLAS FORMACIÓN EMPRESAS
2020, 2021, 2022:
EAL-16, EAL-17, EAL-18, EAL-19, EAL-20, EAL-21, EAL-22, EAL-24, EAL-25

2023, 2024 añadido:
EAL-18a, EAL-18b, EAL-18c

ERRORES MUESTREO
2020, 2021, 2022, 2023, 2024:
EAL-M1


He hagut de fer: pip install openpyxl (és l'engine de pandas per excel més nous)

In [ ]:
file_path = Path("../../Data/raw/EAL/Tablas_EAL_2024.xlsx")

sheets_to_extract = [
    # TABLAS CONTEXTO
    "EAL-C1", "EAL-C2", "EAL-C3",

    # TABLAS FORMACIÓN EMPRESAS
    "EAL-16", "EAL-17", "EAL-18", "EAL-19", "EAL-20", "EAL-21",
    "EAL-22", "EAL-23","EAL-24", "EAL-25",
    "EAL-18a", "EAL-18b", "EAL-18c",

    # ERRORES MUESTREO
    "EAL-M1"
]

# Read selected sheets into a dictionary of DataFrames
dfs = pd.read_excel(
    file_path,
    sheet_name=sheets_to_extract,
    header=None,
    dtype=str
)

# Basic cleaning: remove fully empty rows and columns
for sheet_name, df in dfs.items():
    df = df.dropna(how="all")
    df = df.dropna(axis=1, how="all")
    df = df.reset_index(drop=True)
    dfs[sheet_name] = df

In [ ]:
dfs["EAL-23"]

Código de guardaren csv pasado a markdown por "seguridad" que no se machaquen los que ya hay

# Output folder: ../../Data/processed/EAL/2024
output_folder = file_path.parents[2] / "interim" / "EAL" / "2024"
output_folder.mkdir(parents=True, exist_ok=True)

# Save each dataframe as CSV
for sheet_name, df in dfs.items():
    output_path = output_folder / f"{sheet_name}.csv"

    df.to_csv(
        output_path,
        index=False,
        header=False,
        encoding="utf-8-sig"
    )

print(f"CSV files saved in: {output_folder}")

Definición carpeta entrada, salida y las hojas que queremos extraer

In [ ]:
input_folder = Path("../../Data/raw/EAL")

sheets_to_extract = [
    # TABLAS CONTEXTO
    "EAL-C1", "EAL-C2", "EAL-C3",

    # TABLAS FORMACIÓN EMPRESAS
    "EAL-16", "EAL-17", "EAL-18", "EAL-19", "EAL-20", "EAL-21",
    "EAL-22", "EAL-23", "EAL-24", "EAL-25",
    "EAL-18a", "EAL-18b", "EAL-18c",

    # ERRORES MUESTREO
    "EAL-M1"
]

# Carpeta base de salida: ../../Data/interim/EAL
output_base_folder = input_folder.parents[1] / "pre_processed_2020_2024_" / "EAL_15062026"

In [ ]:
# Procesar todos los archivos .xlsx de la carpeta
for file_path in input_folder.glob("*.xlsx"):

    # Extraer el año de los últimos 4 caracteres del nombre del archivo
    year = file_path.stem[-4:]

    # Comprobar qué hojas existen realmente en el archivo
    excel_file = pd.ExcelFile(file_path)
    available_sheets = [
        sheet for sheet in sheets_to_extract 
        if sheet in excel_file.sheet_names
    ]

    if not available_sheets:
        print(f"No se encontraron hojas válidas en: {file_path.name}")
        continue

    # Leer las hojas seleccionadas en un diccionario de DataFrames
    dfs = pd.read_excel(
        file_path,
        sheet_name=available_sheets,
        header=None,
        dtype=str
    )

    # Limpieza básica: eliminar filas y columnas completamente vacías
    for sheet_name, df in dfs.items():
        df = df.dropna(how="all")
        df = df.dropna(axis=1, how="all")
        df = df.reset_index(drop=True)
        dfs[sheet_name] = df

    # Carpeta de salida por año
    output_folder = output_base_folder / year
    output_folder.mkdir(parents=True, exist_ok=True)

    # Guardar cada DataFrame como CSV
    for sheet_name, df in dfs.items():
        output_path = output_folder / f"{sheet_name}.csv"

        df.to_csv(
            output_path,
            index=False,
            header=False,
            encoding="utf-8-sig"
        )

    print(f"CSV files saved for {file_path.name} in: {output_folder}")